# Colab runs: Phase 6 and memory

The GPU runs of Phase 6 and of the memory extension, done on a Colab L4. Three independent parts:

| Part | What | Runs | Data zip |
|---|---|---|---|
| A | Phase 6 on Moving MNIST | 5 | `vpb-data-mmnist.zip` |
| B | Phase 6 on KTH | 5 | `vpb-data-kth.zip` |
| C | Memory (window / RMT / kNN, 3 seeds) | 9, in 3 waves | `vpb-data-mmnist.zip` |

The code comes in as `vpb-code.zip` (the repo without data) and the data as zips, put in
`/content` or in the Drive folder `vpb`. Checkpoints, logs and results are written to the Drive.
After a disconnect, running Setup, the data cell and the launch cell again skips the finished
runs and resumes the others from their `last.pt`. The "wait" cells print the progress every
5 minutes, and the end of the log if a run fails.

## Setup

In [ ]:
import os, re, time, json, glob, shutil, zipfile, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    VPB = "/content/drive/MyDrive/vpb"
else:
    VPB = "/content/vpb"
for d in ["checkpoints", "logs", "results/phase6", "results/memory"]:
    os.makedirs(f"{VPB}/{d}", exist_ok=True)

# zips can also be kept in the Drive folder vpb once, so a new session does not need them again
for name in ["vpb-code.zip", "vpb-data-mmnist.zip", "vpb-data-kth.zip"]:
    if not os.path.exists(f"/content/{name}") and os.path.exists(f"{VPB}/{name}"):
        shutil.copy(f"{VPB}/{name}", f"/content/{name}")
assert os.path.exists("/content/vpb-code.zip"), "vpb-code.zip not found in /content or in the Drive folder"
with zipfile.ZipFile("/content/vpb-code.zip") as z:
    z.extractall("/content")
REPO = "/content/video-prediction-benchmark"
os.chdir(REPO)
!pip install -q fvcore
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTHONUNBUFFERED"] = "1"   # logs are written line by line, so progress shows up
!python -c "import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())"

In [ ]:
def sh(cmd):
    """Run a shell command in the repo and print the end of its output."""
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=REPO)
    print((r.stdout + r.stderr)[-2500:])
    print(f"[{(time.time() - t0) / 60:.1f} min]")
    if r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")

def launch(jobs, workers=None):
    """jobs: (name, command, result file). Runs whose result already exists are skipped."""
    todo = [j for j in jobs if not os.path.exists(j[2])]
    for name, _, _ in jobs:
        if all(name != t[0] for t in todo):
            print(f"{name}: already done, skipped")
    if not todo:
        return {}
    workers = workers or max(1, (os.cpu_count() - 1) // len(todo))
    procs = {}
    for name, cmd, _ in todo:
        log = open(f"{VPB}/logs/{name}.log", "a")   # a resumed run continues the same log
        p = subprocess.Popen(cmd + ["--num-workers", str(workers)], stdout=log,
                             stderr=subprocess.STDOUT, cwd=REPO)
        procs[name] = (p, time.time())
        time.sleep(5)
    print(f"started {len(procs)} runs, {workers} data workers each")
    return procs

def progress(procs):
    for name, (p, t0) in procs.items():
        text = open(f"{VPB}/logs/{name}.log").read()
        lines = text.strip().splitlines()
        epochs = [l for l in lines if l.startswith("Epoch ")]
        info = epochs[-1][:60] if epochs else (lines[-1][:60] if lines else "starting")
        if epochs and p.poll() is None:
            done, total = map(int, re.match(r"Epoch (\d+)/(\d+)", epochs[-1]).groups())
            resumed = re.findall(r"Resuming after epoch (\d+)", text)
            new = done - (int(resumed[-1]) if resumed else 0)   # epochs done in this session
            if new > 0:
                info += f" | ~{(time.time() - t0) / new * (total - done) / 60:.0f} min left"
        state = "running" if p.poll() is None else ("done" if p.poll() == 0 else f"ERROR {p.poll()}")
        print(f"{name:28s} {state:8s} {info}")

def wait(procs, every=300):
    """Block until all runs are finished, printing the progress every few minutes."""
    while procs and any(p.poll() is None for p, _ in procs.values()):
        progress(procs)
        print("-" * 70)
        time.sleep(every)
    if procs:
        progress(procs)
    for name, (p, _) in procs.items():
        if p.returncode != 0:
            print(f"\n=== {name} failed, end of its log ===")
            print("\n".join(open(f"{VPB}/logs/{name}.log").read().splitlines()[-25:]))

In [ ]:
# Moving MNIST + MNIST (parts A and C)
if os.path.exists("/content/vpb-data-mmnist.zip"):
    with zipfile.ZipFile("/content/vpb-data-mmnist.zip") as z:
        z.extractall(REPO)
    print("Moving MNIST and MNIST taken from vpb-data-mmnist.zip")
else:
    print("vpb-data-mmnist.zip not found, downloading instead (slower)")
    sh("python scripts/download_mmnist.py")
    sh("python -c \"from torchvision import datasets; "
       "datasets.MNIST('data/mnist', train=True, download=True); "
       "datasets.MNIST('data/mnist', train=False, download=True)\"")
print(os.path.getsize("data/moving_mnist/mnist_test_seq.npy") // 10**6, "MB Moving MNIST test file")
print(sorted(os.listdir("data/mnist/MNIST/raw")))

## Part A: Phase 6 on Moving MNIST

`n3` with seeds 1 and 2, then `skip`, `l1` and `dil2` with seed 1.

In [ ]:
jobs_a = [(f"phase6-{v}-seed{s}",
           ["python", "scripts/phase6_train.py", "--variant", v, "--seed", str(s),
            "--data-root", "data", "--ckpt-dir", f"{VPB}/checkpoints",
            "--out-dir", f"{VPB}/results/phase6"],
           f"{VPB}/results/phase6/phase6-{v}-seed{s}.json")
          for v, s in [("n3", 1), ("n3", 2), ("skip", 1), ("l1", 1), ("dil2", 1)]]
procs_a = launch(jobs_a)

In [ ]:
wait(procs_a)   # blocks until the 5 runs are done

## Part B: Phase 6 on KTH

The frames are extracted from the KTH videos (10 to 15 min), with the clip lists of the
original experiments (`splits/kth`). Runs: `n2` with seeds 1 and 2, then `skip`, `l1` and
`dil2` with seed 1.

In [ ]:
if not os.path.exists(f"{REPO}/data/kth/frames/walking"):
    if os.path.exists("/content/vpb-data-kth.zip"):
        with zipfile.ZipFile("/content/vpb-data-kth.zip") as z:
            z.extractall(REPO)
        print("KTH videos taken from vpb-data-kth.zip")
    else:
        print("vpb-data-kth.zip not found, downloading the videos from the KTH site instead")
    sh("python scripts/download_kth.py")        # only unzips if the archives are already there
    sh("python scripts/extract_kth_frames.py")  # avi -> 128x128 grayscale png
for split in ["train", "val", "test"]:
    shutil.copy(f"{REPO}/splits/kth/{split}_clips.txt", f"{REPO}/data/kth/{split}_clips.txt")

# every frame used by the clip lists must exist
missing = 0
for split in ["train", "val", "test"]:
    for line in open(f"{REPO}/data/kth/{split}_clips.txt"):
        if line.strip():
            path, a, b = line.rsplit(" ", 2)
            missing += sum(not os.path.exists(f"{REPO}/data/kth/frames/{path}/{i:04d}.png")
                           for i in range(int(a), int(b)))
print("missing frames:", missing)

# decode the PNGs once into one array: reading PNGs every epoch made the runs CPU-bound
if not os.path.exists(f"{REPO}/data/kth/frames_uint8.npy"):
    sh("python scripts/kth_to_npy.py --kth-root data/kth --workers 8")

In [ ]:
jobs_b = [(f"phase6-kth-{v}-seed{s}",
           ["python", "scripts/phase6_train_kth.py", "--variant", v, "--seed", str(s),
            "--kth-root", "data/kth", "--ckpt-dir", f"{VPB}/checkpoints",
            "--out-dir", f"{VPB}/results/phase6"],
           f"{VPB}/results/phase6/phase6-kth-{v}-seed{s}.json")
          for v, s in [("n2", 1), ("n2", 2), ("skip", 1), ("l1", 1), ("dil2", 1)]]
procs_b = launch(jobs_b)

In [ ]:
wait(procs_b)

## Part C: memory

The timing cell runs one epoch of `knn`, the slowest model (a full run is about 100 times
that). Then wave 1 (seed 1 of each model), the check, and waves 2 and 3.

In [ ]:
shutil.rmtree("/content/tmp_ckpt", ignore_errors=True)   # otherwise a second try would resume
t0 = time.time()
sh("python scripts/train_memory.py --model knn --seed 0 --n-epochs 1 --n-test 100 "
   "--data-root data --ckpt-dir /content/tmp_ckpt --out-dir /content/tmp_out --num-workers 4")
print(f"one epoch + a small evaluation: {(time.time() - t0) / 60:.1f} min")

In [ ]:
def memory_jobs(seed):
    return [(f"memory-{m}-seed{seed}",
             ["python", "scripts/train_memory.py", "--model", m, "--seed", str(seed),
              "--data-root", "data", "--ckpt-dir", f"{VPB}/checkpoints",
              "--out-dir", f"{VPB}/results/memory"],
             f"{VPB}/results/memory/memory-{m}-seed{seed}.json")
            for m in ["window", "rmt", "knn"]]

procs_c1 = launch(memory_jobs(1))
wait(procs_c1)

### Check before waves 2 and 3

Precondition from `MEMORY_RESULTS.md`: for the window model, zone SSIM at k=0 should be at
least 0.1 above k=40. It failed, see `MEMORY_RESULTS.md` for why seeds 2 and 3 were still run.

In [ ]:
for f in sorted(glob.glob(f"{VPB}/results/memory/memory-*-seed1.json")):
    r = json.load(open(f))
    print(r["model"], "zone SSIM by k:", {k: round(v["zone_ssim_avg"], 3) for k, v in r["curve"].items()})

In [ ]:
procs_c2 = launch(memory_jobs(2))
wait(procs_c2)

In [ ]:
procs_c3 = launch(memory_jobs(3))
wait(procs_c3)

## Download the results

The zip goes into `results/` of the local repo (`phase6/` and `memory/`).

In [ ]:
from google.colab import files
print(sorted(os.listdir(f"{VPB}/results/phase6")), sorted(os.listdir(f"{VPB}/results/memory")))
archive = shutil.make_archive("/content/vpb-results", "zip", f"{VPB}/results")
files.download(archive)